# Production-Grade Domain-Specific RAG System

**Submitted by:** Banshidhari Nandi  
**Domain:** Hugging Face Developer Documentation  
**Vector Database:** Milvus Lite  
**Embedding Model:** BAAI/bge-small-en-v1.5  
**Generation Model:** Qwen2-1.5B-Instruct  
**Evaluation Framework:** Opik  

## Objective

This notebook implements an end-to-end Retrieval-Augmented Generation pipeline for answering technical questions about Hugging Face workflows.

The pipeline performs:

1. Documentation loading and structural analysis
2. Sliding-window chunking with source lineage
3. Batched and normalized embedding generation
4. Persistent vector storage using Milvus Lite
5. Semantic top-k retrieval
6. Context-grounded answer generation
7. Hallucination and answer-relevance evaluation
8. Analysis of design decisions, limitations, and production trade-offs

The implementation emphasizes reproducibility, memory-efficient processing, traceable sources, retrieval precision, and grounded answer generation.


## 0. Environment Setup

This section installs and verifies the libraries required for document processing, embedding generation, persistent vector storage, answer generation, and automated evaluation.

Milvus Lite is used as the local persistent vector database. It provides the Milvus API without requiring a separately managed Milvus server.


In [1]:
# Install a tested, bounded dependency set for reproducible execution

%pip install -q \
    "numpy>=1.26,<3" \
    "pandas>=2,<4" \
    "pyarrow>=15,<23" \
    "tqdm>=4.66,<5" \
    "sentence-transformers>=3,<6" \
    "datasets>=3,<5" \
    "transformers>=4.45,<6" \
    "accelerate>=1,<2" \
    "pymilvus>=2.5,<3" \
    "milvus-lite>=2.5,<3" \
    "opik==2.2.77"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.9/158.9 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.1/668.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/

In [2]:
import importlib.util
import sys

required_packages = {
    "pymilvus": "pymilvus",
    "milvus_lite": "milvus_lite",
    "sentence_transformers": "sentence_transformers",
    "transformers": "transformers",
    "opik": "opik"
}

print(f"Python version: {sys.version}")
print("-" * 60)

all_available = True

for package_name, module_name in required_packages.items():
    available = importlib.util.find_spec(module_name) is not None
    all_available = all_available and available

    print(
        f"{package_name:<25}: "
        f"{'Available' if available else 'Missing'}"
    )

assert all_available, (
    "One or more required packages are unavailable."
)

print("\nAll required packages are available.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
------------------------------------------------------------
pymilvus                 : Available
milvus_lite              : Available
sentence_transformers    : Available
transformers             : Available
opik                     : Available

All required packages are available.


In [3]:
# Core imports, reproducibility configuration, and portable paths

import os
import json
import random
import sys
from importlib import metadata as importlib_metadata
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import torch

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IN_COLAB = "google.colab" in sys.modules

# Colab uses /content. Local runs write beneath the repository's artifacts
# directory whether the notebook starts from the repository root or notebooks/.
if IN_COLAB:
    PROJECT_ROOT = Path("/content")
else:
    current_directory = Path.cwd().resolve()
    PROJECT_ROOT = (
        current_directory.parent
        if current_directory.name == "notebooks"
        else current_directory
    )

ARTIFACT_DIR = (
    PROJECT_ROOT
    if IN_COLAB
    else PROJECT_ROOT / "artifacts" / "local_checkpoint"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Optional authentication is read securely from the environment. The token is
# never printed or stored in the notebook.
HF_TOKEN = os.getenv("HF_TOKEN")

print("Environment configured successfully")
print("-" * 60)
print(f"Random seed       : {RANDOM_SEED}")
print(f"Device            : {DEVICE}")
print(f"CUDA available    : {torch.cuda.is_available()}")
print(f"Running in Colab  : {IN_COLAB}")
print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"HF token provided : {HF_TOKEN is not None}")


Environment configured successfully
------------------------------------------------------------
Random seed       : 42
Device            : cuda
CUDA available    : True
Running in Colab  : True
Artifact directory: /content
HF token provided : False


## Stage 1: Chunking the Knowledge Base

### 1.1 Data Loading and Structural Inspection

The knowledge base uses the `m-ric/huggingface_doc` dataset, which contains documentation collected from multiple Hugging Face repositories.

Before designing the chunking strategy, the dataset is inspected for:

- Required fields
- Document count
- Missing values
- Duplicate records
- Source diversity
- Document-length distribution

The complete dataset is retained. No fixed document subset is used because evaluating the full knowledge base provides a more representative test of retrieval quality and scalability.


In [4]:
# Load the complete Hugging Face documentation dataset

import pandas as pd
from datasets import load_dataset

DATASET_NAME = "m-ric/huggingface_doc"

dataset = load_dataset(
    DATASET_NAME,
    split="train",
    token=HF_TOKEN
)

# Convert to a DataFrame for inspection and validation
df = dataset.to_pandas()

required_columns = {"text", "source"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {missing_columns}"
    )

# Retain only the required fields
df = df[["text", "source"]].copy()

# Remove invalid records
df = df.dropna(
    subset=["text", "source"]
)

# Normalize once so chunk offsets map exactly to the stored document text.
df["text"] = df["text"].astype(str).str.strip()
df["source"] = df["source"].astype(str).str.strip()

df = df[
    df["text"].str.strip().ne("")
].reset_index(drop=True)

# Calculate document lengths in characters
df["document_length"] = df["text"].str.len()

print("Dataset loaded successfully")
print("-" * 60)
print(f"Dataset name          : {DATASET_NAME}")
print(f"Number of documents   : {len(df):,}")
print(f"Available columns     : {df.columns.tolist()}")
print(f"Unique source paths   : {df['source'].nunique():,}")
print(
    f"Duplicate records     : "
    f"{df[['text', 'source']].duplicated().sum():,}"
)
print("\nMissing values:")
print(df[["text", "source"]].isna().sum())

print("\nDocument-length statistics:")
print(
    df["document_length"]
    .describe()
    .round(2)
)

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv: reconstructing file:   0%|          |  0.00B / 22.0MB            

huggingface_doc.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset loaded successfully
------------------------------------------------------------
Dataset name          : m-ric/huggingface_doc
Number of documents   : 2,647
Available columns     : ['text', 'source', 'document_length']
Unique source paths   : 2,647
Duplicate records     : 0

Missing values:
text      0
source    0
dtype: int64

Document-length statistics:
count      2647.00
mean       8077.77
std       15850.75
min          21.00
25%        1914.50
50%        4573.00
75%        9247.50
max      371056.00
Name: document_length, dtype: float64


In [5]:
# Convert the validated dataset into document records

documents = (
    df[["text", "source"]]
    .to_dict(orient="records")
)

print(f"Prepared {len(documents):,} documents for chunking.")

print("\nFirst document")
print("-" * 60)
print(f"Source       : {documents[0]['source']}")
print(f"Text length  : {len(documents[0]['text']):,} characters")
print(f"Text preview :\n{documents[0]['text'][:500]}")

# Integrity checks
assert len(documents) == len(df)
assert len(documents) > 0
assert all(document["text"].strip() for document in documents)
assert all(document["source"].strip() for document in documents)

print("\nDataset validation passed.")

Prepared 2,647 documents for chunking.

First document
------------------------------------------------------------
Source       : huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
Text length  : 2,011 characters
Text preview :
Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docum

Dataset validation passed.


### 1.2 Sliding-Window Chunking with Source Lineage

Long documentation pages cannot be passed directly to an embedding model or LLM because they exceed practical context limits and often contain several unrelated concepts.

A fixed-size sliding window is used with:

- **Chunk size:** 1,000 characters
- **Chunk overlap:** 200 characters
- **Step size:** 800 characters

The overlap preserves continuity when an explanation, code example, or sentence crosses a chunk boundary.

Every chunk retains:

- A globally unique `chunk_id`
- Its parent `document_id`
- Its position within the parent document
- The original source path
- Character start and end positions

This metadata provides traceability from a retrieved chunk back to its original documentation page.


In [6]:
# Sliding-window chunking with lineage metadata

from typing import List, Dict


def chunk_document(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[str]:
    """
    Split one document into overlapping character-based chunks.

    Args:
        text: Source document text.
        chunk_size: Maximum characters in each chunk.
        chunk_overlap: Characters shared by consecutive chunks.

    Returns:
        A list of non-empty overlapping chunks.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap cannot be negative.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller than chunk_size."
        )

    if not isinstance(text, str):
        return []

    text = text.strip()

    if not text:
        return []

    if len(text) <= chunk_size:
        return [text]

    chunks = []
    step_size = chunk_size - chunk_overlap
    start = 0

    while start < len(text):
        end = min(
            start + chunk_size,
            len(text)
        )

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        if end == len(text):
            break

        start += step_size

    return chunks


def chunk_all_documents(
    documents: List[Dict],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Dict]:
    """
    Chunk all documents while preserving lineage metadata.

    Args:
        documents: Dictionaries containing text and source.
        chunk_size: Maximum characters in each chunk.
        chunk_overlap: Characters shared by consecutive chunks.

    Returns:
        Chunk dictionaries containing text and source metadata.
    """

    all_chunks = []
    next_chunk_id = 0
    step_size = chunk_size - chunk_overlap

    for document_id, document in enumerate(documents):
        document_text = document.get("text", "")
        source = document.get("source", "unknown")

        document_chunks = chunk_document(
            text=document_text,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, chunk_text in enumerate(document_chunks):
            character_start = chunk_index * step_size
            character_end = min(
                character_start + len(chunk_text),
                len(document_text.strip())
            )

            all_chunks.append({
                "chunk_id": next_chunk_id,
                "document_id": document_id,
                "chunk_index": chunk_index,
                "character_start": character_start,
                "character_end": character_end,
                "source": source,
                "text": chunk_text
            })

            next_chunk_id += 1

    return all_chunks

In [7]:
# Validate sliding-window behavior and edge cases

test_text = "A" * 2500

test_chunks = chunk_document(
    text=test_text,
    chunk_size=1000,
    chunk_overlap=200
)

print("Chunking unit test")
print("-" * 60)
print(f"Original length       : {len(test_text):,}")
print(f"Number of chunks      : {len(test_chunks)}")
print(
    f"Chunk lengths         : "
    f"{[len(chunk) for chunk in test_chunks]}"
)

first_overlap = test_chunks[0][-200:]
second_overlap = test_chunks[1][:200]

print(
    f"First overlap correct : "
    f"{first_overlap == second_overlap}"
)

# Edge-case and behavior checks
assert chunk_document("") == []
assert chunk_document(None) == []
assert chunk_document("Short text", 1000, 200) == ["Short text"]
assert len(test_chunks) == 3
assert [len(chunk) for chunk in test_chunks] == [1000, 1000, 900]
assert first_overlap == second_overlap

print("\nChunking unit tests passed.")

Chunking unit test
------------------------------------------------------------
Original length       : 2,500
Number of chunks      : 3
Chunk lengths         : [1000, 1000, 900]
First overlap correct : True

Chunking unit tests passed.


In [8]:
# Apply chunking to the complete knowledge base

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(
    documents=documents,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunk_lengths = np.array(
    [len(chunk["text"]) for chunk in chunks]
)

unique_chunk_ids = len({
    chunk["chunk_id"]
    for chunk in chunks
})

average_chunks_per_document = (
    len(chunks) / len(documents)
)

print("Full-dataset chunking summary")
print("-" * 60)
print(f"Source documents             : {len(documents):,}")
print(f"Chunks created               : {len(chunks):,}")
print(
    f"Average chunks per document  : "
    f"{average_chunks_per_document:.2f}"
)
print(
    f"Average chunk length         : "
    f"{chunk_lengths.mean():.2f}"
)
print(
    f"Minimum chunk length         : "
    f"{chunk_lengths.min():,}"
)
print(
    f"Maximum chunk length         : "
    f"{chunk_lengths.max():,}"
)
print(f"Unique chunk IDs             : {unique_chunk_ids:,}")

# Integrity validation
assert len(chunks) > len(documents)
assert unique_chunk_ids == len(chunks)
assert chunk_lengths.max() <= CHUNK_SIZE
assert all(chunk["text"].strip() for chunk in chunks)
assert all(chunk["source"].strip() for chunk in chunks)

# Confirm every lineage span reproduces the exact stored chunk text.
assert all(
    documents[chunk["document_id"]]["text"][
        chunk["character_start"]:chunk["character_end"]
    ] == chunk["text"]
    for chunk in chunks
)

required_metadata = {
    "chunk_id",
    "document_id",
    "chunk_index",
    "character_start",
    "character_end",
    "source",
    "text"
}

assert all(
    required_metadata.issubset(chunk.keys())
    for chunk in chunks
)

print("\nFull-dataset chunking validation passed.")

print("\nSample chunk metadata")
print("-" * 60)

sample_chunk = chunks[0]

for field in [
    "chunk_id",
    "document_id",
    "chunk_index",
    "character_start",
    "character_end",
    "source"
]:
    print(f"{field:<18}: {sample_chunk[field]}")

print(f"text preview      : {sample_chunk['text'][:250]}...")

Full-dataset chunking summary
------------------------------------------------------------
Source documents             : 2,647
Chunks created               : 27,434
Average chunks per document  : 10.36
Average chunk length         : 960.10
Minimum chunk length         : 21
Maximum chunk length         : 1,000
Unique chunk IDs             : 27,434

Full-dataset chunking validation passed.

Sample chunk metadata
------------------------------------------------------------
chunk_id          : 0
document_id       : 0
chunk_index       : 0
character_start   : 0
character_end     : 1000
source            : huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
text preview      : Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english]...


## Stage 2: Vectorizing and Storing Knowledge

### 2.1 Batched and Normalized Embedding Generation

Each documentation chunk is converted into a dense vector using `BAAI/bge-small-en-v1.5`.

This model was selected because it:

- Is designed for English semantic retrieval
- Produces compact 384-dimensional embeddings
- Provides a practical balance between retrieval quality, latency, and memory usage
- Supports normalized embeddings suitable for inner-product similarity search

Embeddings are generated in batches to avoid loading all intermediate tensors into memory simultaneously. Each vector is L2-normalized so that Milvus inner-product similarity is equivalent to cosine similarity.

Document chunks are embedded without an instruction prefix. Retrieval queries will use the BGE query instruction recommended for asymmetric search.


In [9]:
# Load the production embedding model

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

# The chunks are approximately 1,000 characters and fit within
# the model's supported token limit.
embedding_model.max_seq_length = 512

EMBEDDING_DIM = (
    embedding_model.get_sentence_embedding_dimension()
)

print("Embedding model loaded successfully")
print("-" * 60)
print(f"Model name          : {EMBEDDING_MODEL_NAME}")
print(f"Execution device    : {embedding_model.device}")
print(f"Embedding dimension : {EMBEDDING_DIM}")
print(f"Maximum sequence    : {embedding_model.max_seq_length}")

assert EMBEDDING_DIM == 384

print("\nEmbedding model validation passed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully
------------------------------------------------------------
Model name          : BAAI/bge-small-en-v1.5
Execution device    : cuda:0
Embedding dimension : 384
Maximum sequence    : 512

Embedding model validation passed.


/tmp/ipykernel_2936/1353142000.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [10]:
# Memory-efficient batched embedding generation

from typing import List
from tqdm.auto import tqdm


def generate_embeddings(
    texts: List[str],
    model: SentenceTransformer,
    batch_size: int = 32
) -> np.ndarray:
    """
    Convert texts into normalized dense embeddings in batches.

    Args:
        texts: Texts to encode.
        model: SentenceTransformer embedding model.
        batch_size: Number of texts processed per batch.

    Returns:
        A float32 NumPy array containing normalized embeddings.
    """

    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero.")

    if not texts:
        return np.empty(
            (0, EMBEDDING_DIM),
            dtype=np.float32
        )

    all_embeddings = np.empty(
        (len(texts), EMBEDDING_DIM),
        dtype=np.float32
    )

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Generating BGE embeddings"
    ):
        end = min(
            start + batch_size,
            len(texts)
        )

        text_batch = texts[start:end]

        batch_embeddings = model.encode(
            text_batch,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False
        )

        batch_embeddings = np.asarray(
            batch_embeddings,
            dtype=np.float32
        )

        expected_shape = (
            len(text_batch),
            EMBEDDING_DIM
        )

        if batch_embeddings.shape != expected_shape:
            raise ValueError(
                f"Unexpected shape {batch_embeddings.shape}; "
                f"expected {expected_shape}."
            )

        all_embeddings[start:end] = batch_embeddings

    return all_embeddings

In [11]:
# Validate embedding generation on a small sample

test_texts = [
    "How do I fine-tune a transformer model?",
    "How can I load a Hugging Face dataset?",
    "What is Gradio used for?"
]

test_embeddings = generate_embeddings(
    texts=test_texts,
    model=embedding_model,
    batch_size=3
)

test_norms = np.linalg.norm(
    test_embeddings,
    axis=1
)

print("Embedding unit test")
print("-" * 60)
print(f"Input texts         : {len(test_texts)}")
print(f"Embedding shape     : {test_embeddings.shape}")
print(f"Data type           : {test_embeddings.dtype}")
print(f"Vector norms        : {test_norms}")

assert test_embeddings.shape == (3, EMBEDDING_DIM)
assert test_embeddings.dtype == np.float32
assert np.allclose(test_norms, 1.0, atol=1e-5)

print("\nEmbedding unit test passed.")

Generating BGE embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding unit test
------------------------------------------------------------
Input texts         : 3
Embedding shape     : (3, 384)
Data type           : float32
Vector norms        : [1. 1. 1.]

Embedding unit test passed.


In [12]:
# Generate BGE embeddings for the complete chunk collection

import time

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

# A moderate batch size reduces CPU memory pressure.
# Increase to 64 if running on a T4 GPU.
EMBEDDING_BATCH_SIZE = (
    64 if torch.cuda.is_available() else 32
)

print("Starting full-dataset embedding generation")
print("-" * 60)
print(f"Chunks to embed      : {len(chunk_texts):,}")
print(f"Batch size           : {EMBEDDING_BATCH_SIZE}")
print(f"Execution device     : {embedding_model.device}")

embedding_start_time = time.perf_counter()

embeddings = generate_embeddings(
    texts=chunk_texts,
    model=embedding_model,
    batch_size=EMBEDDING_BATCH_SIZE
)

embedding_elapsed_time = (
    time.perf_counter() - embedding_start_time
)

# Persist a checkpoint in the configured runtime artifact directory.
EMBEDDING_CHECKPOINT = ARTIFACT_DIR / "bge_chunk_embeddings.npy"

np.save(
    EMBEDDING_CHECKPOINT,
    embeddings
)

# Validate dimensions and normalization
embedding_norms = np.linalg.norm(
    embeddings,
    axis=1
)

nonzero_vectors = embedding_norms > 0
zero_vector_count = int(
    (~nonzero_vectors).sum()
)

print("\nFull embedding summary")
print("-" * 60)
print(f"Chunks embedded      : {len(embeddings):,}")
print(f"Embedding shape      : {embeddings.shape}")
print(f"Embedding dimension  : {embeddings.shape[1]}")
print(f"Data type            : {embeddings.dtype}")
print(
    f"Memory usage         : "
    f"{embeddings.nbytes / (1024 ** 2):.2f} MB"
)
print(
    f"Processing time      : "
    f"{embedding_elapsed_time:.2f} seconds"
)
print(f"Nonzero vectors      : {nonzero_vectors.sum():,}")
print(f"Zero vectors         : {zero_vector_count:,}")
print(
    f"Minimum vector norm  : "
    f"{embedding_norms.min():.6f}"
)
print(
    f"Maximum vector norm  : "
    f"{embedding_norms.max():.6f}"
)
print(
    f"Average vector norm  : "
    f"{embedding_norms.mean():.6f}"
)
print(f"Checkpoint           : {EMBEDDING_CHECKPOINT}")

# Integrity validation
assert embeddings.shape == (
    len(chunks),
    EMBEDDING_DIM
)

assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()
assert zero_vector_count == 0

assert np.allclose(
    embedding_norms,
    1.0,
    atol=1e-5
)

print("\nFull-dataset embedding validation passed.")

Starting full-dataset embedding generation
------------------------------------------------------------
Chunks to embed      : 27,434
Batch size           : 64
Execution device     : cuda:0


Generating BGE embeddings:   0%|          | 0/429 [00:00<?, ?it/s]


Full embedding summary
------------------------------------------------------------
Chunks embedded      : 27,434
Embedding shape      : (27434, 384)
Embedding dimension  : 384
Data type            : float32
Memory usage         : 40.19 MB
Processing time      : 288.70 seconds
Nonzero vectors      : 27,434
Zero vectors         : 0
Minimum vector norm  : 1.000000
Maximum vector norm  : 1.000000
Average vector norm  : 1.000000
Checkpoint           : /content/bge_chunk_embeddings.npy

Full-dataset embedding validation passed.


### 2.3 Persistent Vector Storage with Milvus Lite

The normalized chunk embeddings are stored in a persistent Milvus Lite database.

The collection uses:

- `id` as the integer primary key
- `vector` as a 384-dimensional `FLOAT_VECTOR`
- Inner Product (`IP`) as the similarity metric
- Strong consistency for deterministic reads after writes
- Dynamic fields for chunk text and lineage metadata

Because every embedding is L2-normalized, inner-product ranking is equivalent to cosine-similarity ranking.

Milvus Lite stores the collection in a local database file, allowing the vector index and inserted records to persist beyond the client object's lifetime.


In [13]:
# Initialize a fresh persistent Milvus Lite client

from pathlib import Path
from pymilvus import MilvusClient

# Use a portable persistent path for Colab and local execution.
MILVUS_DB_PATH = str(ARTIFACT_DIR / "hf_docs_milvus_clean.db")
COLLECTION_NAME = "hf_documentation"

milvus_client = MilvusClient(
    uri=MILVUS_DB_PATH
)

print("Milvus Lite client initialized")
print("-" * 60)
print(f"Database path   : {MILVUS_DB_PATH}")
print(f"Database exists : {Path(MILVUS_DB_PATH).exists()}")
print(f"Collection name : {COLLECTION_NAME}")

/usr/local/lib/python3.13/dist-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Milvus Lite client initialized
------------------------------------------------------------
Database path   : /content/hf_docs_milvus_clean.db
Database exists : True
Collection name : hf_documentation


In [14]:
# Create a Milvus collection configured for normalized BGE vectors


def setup_milvus_collection(
    client: MilvusClient,
    collection_name: str,
    embedding_dim: int
) -> None:
    """
    Create a persistent Milvus collection.

    Existing collections with the same name are removed to make
    notebook reruns idempotent and prevent duplicate ingestion.

    Args:
        client: Initialized MilvusClient.
        collection_name: Collection to create.
        embedding_dim: Dimension of the embedding vectors.
    """

    if embedding_dim <= 0:
        raise ValueError(
            "embedding_dim must be greater than zero."
        )

    if client.has_collection(
        collection_name=collection_name
    ):
        client.drop_collection(
            collection_name=collection_name
        )

        print(
            f"Dropped existing collection: "
            f"{collection_name}"
        )

    client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        primary_field_name="id",
        id_type="int",
        vector_field_name="vector",
        metric_type="IP",
        auto_id=False,
        consistency_level="Strong",
        enable_dynamic_field=True
    )

    if not client.has_collection(
        collection_name=collection_name
    ):
        raise RuntimeError(
            f"Collection {collection_name} was not created."
        )

    print(
        f"Created collection: {collection_name}"
    )

In [15]:
# Create and inspect the Milvus collection

setup_milvus_collection(
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_dim=EMBEDDING_DIM
)

collection_details = (
    milvus_client.describe_collection(
        collection_name=COLLECTION_NAME
    )
)

index_names = milvus_client.list_indexes(
    collection_name=COLLECTION_NAME
)

print("\nMilvus collection summary")
print("-" * 60)
print(f"Database file       : {MILVUS_DB_PATH}")
print(
    f"Database exists     : "
    f"{Path(MILVUS_DB_PATH).exists()}"
)
print(f"Collection name     : {COLLECTION_NAME}")
print(f"Embedding dimension : {EMBEDDING_DIM}")
print("Similarity metric   : IP")
print("Consistency level   : Strong")
print(f"Available indexes   : {index_names}")

print("\nCollection details:")
print(collection_details)

# Validate collection persistence and configuration
assert Path(MILVUS_DB_PATH).exists()

assert milvus_client.has_collection(
    collection_name=COLLECTION_NAME
)

assert len(index_names) > 0

print(
    "\nPersistent Milvus collection "
    "validation passed."
)

Created collection: hf_documentation

Milvus collection summary
------------------------------------------------------------
Database file       : /content/hf_docs_milvus_clean.db
Database exists     : True
Collection name     : hf_documentation
Embedding dimension : 384
Similarity metric   : IP
Consistency level   : Strong
Available indexes   : ['vector']

Collection details:
{'collection_name': 'hf_documentation', 'auto_id': False, 'num_shards': 0, 'description': '', 'fields': [{'field_id': 100, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 101, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 0, 'enable_dynamic_field': True, 'enable_namespace': False}

Persistent Milvus collection validation passed.


### 2.4 Batched Ingestion and Full Record Verification

Every insertion batch must be acknowledged by Milvus. After flushing, the verification step checks three independent signals: the sum of batch acknowledgements, the persisted collection row count, and batched retrieval of every expected primary key. It also rejects missing, unexpected, or duplicate IDs and compares stored lineage metadata with representative source chunks.


In [ ]:
# Insert embeddings and lineage metadata into Milvus in batches


def insert_data_to_milvus(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    embeddings: np.ndarray,
    batch_size: int = 256
) -> int:
    """Insert every chunk and validate each Milvus batch acknowledgement.

    Args:
        client: Initialized MilvusClient.
        collection_name: Target collection.
        chunks: Chunk dictionaries containing text and lineage.
        embeddings: Normalized embedding matrix.
        batch_size: Records inserted per Milvus request.

    Returns:
        Sum of records acknowledged across all insertion batches.
    """
    if batch_size <= 0:
        raise ValueError("batch_size must be greater than zero.")

    if len(chunks) != len(embeddings):
        raise ValueError(
            "The number of chunks must match the number of embeddings."
        )

    if embeddings.ndim != 2 or embeddings.shape[1] != EMBEDDING_DIM:
        raise ValueError(
            f"Expected embedding shape (n, {EMBEDDING_DIM}), "
            f"but received {embeddings.shape}."
        )

    total_inserted = 0

    for start in tqdm(
        range(0, len(chunks), batch_size),
        desc="Inserting records into Milvus"
    ):
        end = min(start + batch_size, len(chunks))
        batch_records = []

        for index in range(start, end):
            chunk = chunks[index]
            batch_records.append({
                "id": int(chunk["chunk_id"]),
                "vector": embeddings[index].tolist(),
                "text": chunk["text"],
                "source": chunk["source"],
                "document_id": int(chunk["document_id"]),
                "chunk_index": int(chunk["chunk_index"]),
                "character_start": int(chunk["character_start"]),
                "character_end": int(chunk["character_end"])
            })

        result = client.insert(
            collection_name=collection_name,
            data=batch_records
        )

        inserted_in_batch = int(result.get("insert_count", 0))
        expected_in_batch = len(batch_records)

        if inserted_in_batch != expected_in_batch:
            raise RuntimeError(
                f"Batch {start // batch_size + 1} insertion mismatch: "
                f"expected {expected_in_batch}, acknowledged "
                f"{inserted_in_batch}."
            )

        # When Milvus returns inserted primary keys, validate those as a
        # fourth immediate acknowledgement signal.
        response_ids = result.get("ids")
        if response_ids is not None:
            expected_batch_ids = {
                int(record["id"])
                for record in batch_records
            }
            returned_batch_ids = {
                int(record_id)
                for record_id in response_ids
            }

            if returned_batch_ids != expected_batch_ids:
                raise RuntimeError(
                    "Milvus returned primary keys that do not match "
                    f"batch {start // batch_size + 1}."
                )

        total_inserted += inserted_in_batch

    client.flush(collection_name=collection_name)
    return total_inserted


def verify_all_milvus_records(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    acknowledged_count: int,
    verification_batch_size: int = 1000
) -> Dict:
    """Prove that every expected chunk ID and its metadata persisted.

    Verification is independent of the insertion loop: all expected primary
    keys are read back from Milvus in batches and compared as sets. Three
    representative rows are then compared field-by-field with source chunks.

    Returns:
        Counts used for the explicit Task 2.4 confirmation report.
    """
    if verification_batch_size <= 0:
        raise ValueError(
            "verification_batch_size must be greater than zero."
        )

    expected_ids = [int(chunk["chunk_id"]) for chunk in chunks]
    expected_id_set = set(expected_ids)

    if len(expected_id_set) != len(expected_ids):
        raise ValueError("Expected chunk IDs are not unique.")

    collection_stats = client.get_collection_stats(
        collection_name=collection_name
    )
    stored_row_count = int(collection_stats.get("row_count", 0))

    persisted_ids = []

    for start in tqdm(
        range(0, len(expected_ids), verification_batch_size),
        desc="Verifying persisted Milvus IDs"
    ):
        id_batch = expected_ids[start:start + verification_batch_size]
        stored_records = client.get(
            collection_name=collection_name,
            ids=id_batch,
            output_fields=["id"]
        )
        persisted_ids.extend(
            int(record["id"])
            for record in stored_records
        )

    persisted_id_set = set(persisted_ids)
    missing_ids = sorted(expected_id_set - persisted_id_set)
    unexpected_ids = sorted(persisted_id_set - expected_id_set)
    duplicate_id_count = len(persisted_ids) - len(persisted_id_set)

    sample_positions = sorted({0, len(chunks) // 2, len(chunks) - 1})
    sample_ids = [int(chunks[position]["chunk_id"]) for position in sample_positions]
    sampled_records = client.get(
        collection_name=collection_name,
        ids=sample_ids,
        output_fields=[
            "id",
            "text",
            "source",
            "document_id",
            "chunk_index",
            "character_start",
            "character_end"
        ]
    )
    sampled_by_id = {
        int(record["id"]): record
        for record in sampled_records
    }

    metadata_fields = [
        "text",
        "source",
        "document_id",
        "chunk_index",
        "character_start",
        "character_end"
    ]

    metadata_matches = 0
    for position in sample_positions:
        expected_chunk = chunks[position]
        chunk_id = int(expected_chunk["chunk_id"])
        stored_record = sampled_by_id.get(chunk_id)

        if stored_record is None:
            raise RuntimeError(
                f"Metadata verification could not retrieve chunk {chunk_id}."
            )

        if not all(
            stored_record[field] == expected_chunk[field]
            for field in metadata_fields
        ):
            raise RuntimeError(
                f"Stored metadata does not match chunk {chunk_id}."
            )

        metadata_matches += 1

    expected_count = len(chunks)

    assert acknowledged_count == expected_count
    assert stored_row_count == expected_count
    assert len(persisted_ids) == expected_count
    assert not missing_ids
    assert not unexpected_ids
    assert duplicate_id_count == 0
    assert metadata_matches == len(sample_positions)

    return {
        "expected_count": expected_count,
        "acknowledged_count": acknowledged_count,
        "stored_row_count": stored_row_count,
        "retrieved_id_count": len(persisted_ids),
        "missing_id_count": len(missing_ids),
        "unexpected_id_count": len(unexpected_ids),
        "duplicate_id_count": duplicate_id_count,
        "metadata_samples_checked": metadata_matches
    }


In [ ]:
# Insert the complete knowledge base and confirm every persisted record

if not milvus_client.has_collection(
    collection_name=COLLECTION_NAME
):
    raise RuntimeError(
        "The Milvus collection does not exist. "
        "Run the collection-setup cell first."
    )

INSERT_BATCH_SIZE = 256
VERIFICATION_BATCH_SIZE = 1000

insertion_start_time = time.perf_counter()

inserted_count = insert_data_to_milvus(
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    chunks=chunks,
    embeddings=embeddings,
    batch_size=INSERT_BATCH_SIZE
)

insertion_elapsed_time = time.perf_counter() - insertion_start_time

verification_report = verify_all_milvus_records(
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    chunks=chunks,
    acknowledged_count=inserted_count,
    verification_batch_size=VERIFICATION_BATCH_SIZE
)

# Retain this checkpoint variable for the final reproducibility manifest.
stored_row_count = verification_report["stored_row_count"]

print("\nTask 2.4 - Milvus ingestion and persistence confirmation")
print("-" * 60)
print(
    f"Records expected            : "
    f"{verification_report['expected_count']:,}"
)
print(
    f"Batch acknowledgements      : "
    f"{verification_report['acknowledged_count']:,}"
)
print(
    f"Persisted collection rows   : "
    f"{verification_report['stored_row_count']:,}"
)
print(
    f"Primary keys read back      : "
    f"{verification_report['retrieved_id_count']:,}"
)
print(
    f"Missing primary keys        : "
    f"{verification_report['missing_id_count']}"
)
print(
    f"Unexpected primary keys     : "
    f"{verification_report['unexpected_id_count']}"
)
print(
    f"Duplicate primary keys      : "
    f"{verification_report['duplicate_id_count']}"
)
print(
    f"Metadata samples confirmed  : "
    f"{verification_report['metadata_samples_checked']}"
)
print(f"Insertion batch size        : {INSERT_BATCH_SIZE}")
print(f"Verification batch size     : {VERIFICATION_BATCH_SIZE}")
print(f"Insertion time              : {insertion_elapsed_time:.2f} seconds")
print(f"Database path               : {MILVUS_DB_PATH}")

print(
    "\nCONFIRMED: every expected chunk record was inserted, "
    "persisted, and retrieved successfully from Milvus."
)


## Stage 3: Retrieving Relevant Context

A natural-language query is transformed into the same 384-dimensional vector space as the documentation chunks.

For asymmetric BGE retrieval:

- Documentation chunks are encoded as passages without a prefix.
- User questions are encoded with the BGE retrieval instruction.
- The query vector is L2-normalized.
- Milvus ranks stored vectors using inner product.
- The highest-scoring `top_k` chunks are returned with their source and lineage metadata.

The returned similarity score is used to inspect retrieval confidence and diagnose weak or out-of-domain matches.


In [18]:
# Retrieve semantically relevant chunks from Milvus


def retrieve_documents(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5
) -> List[Dict]:
    """
    Retrieve the highest-scoring documentation chunks.

    Args:
        query: Natural-language user question.
        client: Initialized MilvusClient.
        collection_name: Milvus collection to search.
        embedding_model: BGE SentenceTransformer model.
        top_k: Maximum number of results to return.

    Returns:
        Ranked dictionaries containing text, source,
        lineage metadata, and similarity score.
    """

    if not isinstance(query, str) or not query.strip():
        raise ValueError(
            "query must be a non-empty string."
        )

    if top_k <= 0:
        raise ValueError(
            "top_k must be greater than zero."
        )

    # BGE recommends an instruction for retrieval queries
    instructed_query = (
        QUERY_INSTRUCTION + query.strip()
    )

    query_embedding = embedding_model.encode(
        [instructed_query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    if query_embedding.shape != (
        1,
        EMBEDDING_DIM
    ):
        raise ValueError(
            f"Unexpected query embedding shape: "
            f"{query_embedding.shape}"
        )

    search_results = client.search(
        collection_name=collection_name,
        data=[query_embedding[0].tolist()],
        limit=top_k,
        search_params={
            "metric_type": "IP",
            "params": {}
        },
        output_fields=[
            "text",
            "source",
            "document_id",
            "chunk_index",
            "character_start",
            "character_end"
        ]
    )

    retrieved_documents = []

    # One query produces one result list
    for rank, hit in enumerate(
        search_results[0],
        start=1
    ):
        entity = hit.get("entity", {})

        retrieved_documents.append({
            "rank": rank,
            "id": int(hit["id"]),
            "score": float(hit["distance"]),
            "text": entity.get("text", ""),
            "source": entity.get(
                "source",
                "unknown"
            ),
            "document_id": entity.get(
                "document_id"
            ),
            "chunk_index": entity.get(
                "chunk_index"
            ),
            "character_start": entity.get(
                "character_start"
            ),
            "character_end": entity.get(
                "character_end"
            )
        })

    return retrieved_documents

In [19]:
# Validate semantic retrieval

test_query = (
    "How do I fine-tune a transformer model?"
)

RETRIEVAL_TOP_K = 5

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=RETRIEVAL_TOP_K
)

print("Semantic retrieval test")
print("-" * 60)
print(f"Query             : {test_query}")
print(f"Requested top-k   : {RETRIEVAL_TOP_K}")
print(f"Retrieved results : {len(retrieved)}")

for document in retrieved:
    print(
        f"\nRank {document['rank']} | "
        f"Score: {document['score']:.4f}"
    )
    print(f"Chunk ID : {document['id']}")
    print(f"Source   : {document['source']}")
    print(
        f"Text     : "
        f"{document['text'][:350]}..."
    )

# Retrieval integrity checks
assert len(retrieved) == RETRIEVAL_TOP_K
assert all(document["text"] for document in retrieved)
assert all(document["source"] for document in retrieved)

retrieval_scores = [
    document["score"]
    for document in retrieved
]

assert retrieval_scores == sorted(
    retrieval_scores,
    reverse=True
)

query_norm = np.linalg.norm(
    embedding_model.encode(
        [QUERY_INSTRUCTION + test_query],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]
)

assert np.isclose(
    query_norm,
    1.0,
    atol=1e-5
)

print("\nSemantic retrieval validation passed.")

Semantic retrieval test
------------------------------------------------------------
Query             : How do I fine-tune a transformer model?
Requested top-k   : 5
Retrieved results : 5

Rank 1 | Score: 0.7828
Chunk ID : 9430
Source   : huggingface/blog/blob/main/cv_state.md
Text     : our own models

While being able to use a model for off-the-shelf inference is a great way to get started, fine-tuning is where the community gets the most benefits. This is especially true when your datasets are custom, and you’re not getting good performance out of the pre-trained models.

Transformers provides a [Trainer API](https://huggingface...

Rank 2 | Score: 0.7777
Chunk ID : 21362
Source   : huggingface/transformers/blob/main/docs/source/en/training.md
Text     : 

There are significant benefits to using a pretrained model. It reduces computation costs, your carbon footprint, and allows you to use state-of-the-art models without having to train one from scratch. 🤗 Transformers provides acce

## Stage 4: Grounded Answer Generation

The retrieved documentation chunks are supplied to an instruction-tuned language model to generate the final answer.

This implementation uses `Qwen/Qwen2-1.5B-Instruct` because it offers a practical balance between:

- Instruction-following quality
- Colab GPU memory requirements
- Generation latency
- Local reproducibility
- Grounded technical synthesis

The generation pipeline uses deterministic decoding rather than random sampling. This improves reproducibility and reduces unsupported variations between runs.

Hallucination controls include:

1. Answering only from retrieved context
2. Explicitly refusing when context is insufficient
3. Requiring source references in the answer
4. Applying a minimum retrieval-confidence threshold
5. Returning the retrieved evidence alongside every answer


### 4.1 Generation-model candidates

Candidate considered: [Phi-3 Mini 4K Instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct).


Candidate considered: [Phi-3.5 Mini Instruct](https://huggingface.co/microsoft/Phi-3.5-mini-instruct).


Selected for this implementation: [Qwen2 1.5B Instruct](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct), primarily for lower Colab memory use and deterministic local execution.


In [20]:
# Load the instruction-tuned generation model

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

LLM_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

print("Loading generation model")
print("-" * 60)
print(f"Model  : {LLM_MODEL_NAME}")
print(f"Device : {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True
)

model_loading_arguments = {
    "trust_remote_code": True,
    "low_cpu_mem_usage": True
}

if torch.cuda.is_available():
    model_loading_arguments.update({
        "torch_dtype": torch.float16,
        "device_map": "auto"
    })
else:
    model_loading_arguments.update({
        "torch_dtype": torch.float32
    })

generation_model = (
    AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        **model_loading_arguments
    )
)

generation_model.eval()

# Ensure a valid padding token exists
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = (
        tokenizer.eos_token_id
    )

generator = pipeline(
    task="text-generation",
    model=generation_model,
    tokenizer=tokenizer
)

model_device = next(
    generation_model.parameters()
).device

print("\nGeneration model loaded successfully")
print("-" * 60)
print(f"Model name     : {LLM_MODEL_NAME}")
print(f"Model device   : {model_device}")
print(
    f"Model data type: "
    f"{next(generation_model.parameters()).dtype}"
)
print(
    f"EOS token ID   : "
    f"{tokenizer.eos_token_id}"
)
print(
    f"PAD token ID   : "
    f"{tokenizer.pad_token_id}"
)

assert generator is not None
assert tokenizer.eos_token_id is not None

print("\nGeneration model validation passed.")

Loading generation model
------------------------------------------------------------
Model  : Qwen/Qwen2-1.5B-Instruct
Device : cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Generation model loaded successfully
------------------------------------------------------------
Model name     : Qwen/Qwen2-1.5B-Instruct
Model device   : cuda:0
Model data type: torch.float16
EOS token ID   : 151645
PAD token ID   : 151643

Generation model validation passed.


In [21]:
# Grounded RAG prompt configuration

SYSTEM_PROMPT = """
You are a technical support assistant for Hugging Face developer workflows.

Follow these rules strictly:

1. Answer only from the supplied documentation context.
2. Do not introduce facts that are absent from the context.
3. Cite supporting evidence using [Source 1], [Source 2], and so on.
4. If the context is insufficient, respond exactly:
   "I don't have enough information in the provided documentation to answer this question."
5. If sources conflict, explicitly mention the conflict.
6. Provide a concise, technically accurate answer.
7. Do not claim that you executed code or verified external resources.
""".strip()


def build_grounded_context(
    retrieved_docs: List[Dict]
) -> str:
    """
    Format retrieved chunks as numbered evidence blocks.
    """

    context_blocks = []

    for source_number, document in enumerate(
        retrieved_docs,
        start=1
    ):
        context_blocks.append(
            f"[Source {source_number}]\n"
            f"Source path: {document['source']}\n"
            f"Chunk ID: {document['id']}\n"
            f"Similarity score: {document['score']:.4f}\n"
            f"Content:\n{document['text']}"
        )

    return "\n\n".join(context_blocks)


print("Grounded prompt configuration created.")

Grounded prompt configuration created.


In [ ]:
# Grounded generation with citation validation and deterministic decoding

import re


def generate_answer(
    query: str,
    retrieved_docs: List[Dict],
    generator,
    max_new_tokens: int = 300,
    minimum_retrieval_score: float = 0.65,
    max_context_documents: int = 2
) -> Dict:
    """
    Generate a concise answer using retrieved documentation only.

    The function applies:
    - Retrieval-score filtering
    - Context-document limits
    - Deterministic decoding
    - Citation validation
    - One citation-repair attempt
    - Deterministic citation fallback that preserves grounded answers

    Args:
        query: User's natural-language question.
        retrieved_docs: Ranked documents returned from Milvus.
        generator: Hugging Face text-generation pipeline.
        max_new_tokens: Maximum generated tokens.
        minimum_retrieval_score: Minimum acceptable retrieval score.
        max_context_documents: Maximum evidence chunks given to the LLM.

    Returns:
        Structured RAG response and diagnostic metadata.
    """

    insufficient_information_message = (
        "I don't have enough information in the provided "
        "documentation to answer this question."
    )

    # --------------------------------------------------------
    # Input validation
    # --------------------------------------------------------

    if not isinstance(query, str) or not query.strip():
        raise ValueError(
            "query must be a non-empty string."
        )

    if max_new_tokens <= 0:
        raise ValueError(
            "max_new_tokens must be greater than zero."
        )

    if max_context_documents <= 0:
        raise ValueError(
            "max_context_documents must be greater than zero."
        )

    # --------------------------------------------------------
    # Retrieval-confidence filtering
    # --------------------------------------------------------

    qualified_documents = [
        document
        for document in retrieved_docs
        if document.get("score", 0.0)
        >= minimum_retrieval_score
    ]

    # Limit lower-ranked or overly specific chunks
    accepted_documents = qualified_documents[
        :max_context_documents
    ]

    best_retrieval_score = max(
        (
            document.get("score", 0.0)
            for document in retrieved_docs
        ),
        default=0.0
    )

    # Reject queries when no retrieved evidence meets the threshold
    if not accepted_documents:
        return {
            "query": query,
            "answer": insufficient_information_message,
            "context": "",
            "retrieved_docs": retrieved_docs,
            "used_docs": [],
            "best_retrieval_score": best_retrieval_score,
            "guardrail_triggered": True,
            "guardrail_reason": (
                "retrieval_score_below_threshold"
            ),
            "citation_retry_performed": False,
            "citation_fallback_used": False,
            "citation_repair_method": None,
            "citation_validation_passed": False
        }

    # --------------------------------------------------------
    # Context and prompt construction
    # --------------------------------------------------------

    context = build_grounded_context(
        accepted_documents
    )

    user_prompt = f"""
Answer the question using only the numbered documentation sources.

<context>
{context}
</context>

<question>
{query.strip()}
</question>

Mandatory response rules:

1. Begin with a direct answer to the exact question.
2. Provide no more than 3 short supporting steps or points.
3. Include only steps that are necessary to answer the question.
4. Exclude optional publishing, repository-management, collaboration,
   deployment, or administration details unless explicitly requested.
5. Every factual claim and procedural step must be explicitly supported
   by at least one supplied documentation source.
6. Cite supporting evidence using labels such as [Source 1].
7. Use only APIs, parameters, examples, and claims found in the context.
8. Do not create hyperlinks for source labels.
9. Do not repeat the answer in a separate summary.
10. Do not infer missing information from general knowledge.
11. If the context is insufficient, return exactly:
    "I don't have enough information in the provided documentation to answer this question."
""".strip()

    # --------------------------------------------------------
    # Deterministic model execution
    # --------------------------------------------------------

    def run_model(
        prompt_text: str
    ) -> str:
        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt_text
            }
        ]

        formatted_prompt = (
            generator.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )

        model_inputs = generator.tokenizer(
            formatted_prompt,
            return_tensors="pt"
        )

        model_device = next(
            generator.model.parameters()
        ).device

        model_inputs = {
            name: tensor.to(model_device)
            for name, tensor in model_inputs.items()
        }

        input_token_count = (
            model_inputs["input_ids"].shape[1]
        )

        with torch.inference_mode():
            generated_ids = (
                generator.model.generate(
                    **model_inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=1.08,
                    pad_token_id=(
                        generator.tokenizer.pad_token_id
                    ),
                    eos_token_id=(
                        generator.tokenizer.eos_token_id
                    )
                )
            )

        new_token_ids = generated_ids[
            0,
            input_token_count:
        ]

        generated_text = (
            generator.tokenizer.decode(
                new_token_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )
        )

        return generated_text.strip()

    # Convert Markdown citation links such as
    # [Source 1](invented-url) into the approved [Source 1] form.
    def normalize_source_citations(
        generated_answer: str
    ) -> str:
        return re.sub(
            r"\[(Source\s+\d+)\]\([^)]+\)",
            r"[\1]",
            generated_answer
        )

    # --------------------------------------------------------
    # First generation attempt
    # --------------------------------------------------------

    answer = normalize_source_citations(
        run_model(user_prompt)
    )

    if not answer:
        answer = insufficient_information_message

    # Keep the first useful answer so a citation-only retry cannot turn a
    # supported response into a false insufficient-information refusal.
    initial_answer = answer

    valid_source_numbers = set(
        range(
            1,
            len(accepted_documents) + 1
        )
    )

    def extract_cited_source_numbers(
        generated_answer: str
    ) -> set:
        return {
            int(source_number)
            for source_number in re.findall(
                r"\[Source\s+(\d+)\]",
                generated_answer
            )
        }

    cited_source_numbers = (
        extract_cited_source_numbers(answer)
    )

    citation_validation_passed = bool(
        cited_source_numbers
        and cited_source_numbers.issubset(
            valid_source_numbers
        )
    )

    citation_retry_performed = False

    # --------------------------------------------------------
    # Retry once when citations are absent or invalid
    # --------------------------------------------------------

    if (
        answer != insufficient_information_message
        and not citation_validation_passed
    ):
        citation_retry_performed = True

        allowed_source_labels = ", ".join(
            f"[Source {source_number}]"
            for source_number in sorted(
                valid_source_numbers
            )
        )

        retry_prompt = (
            user_prompt
            + "\n\nYour previous response did not include valid "
              "source citations. Rewrite the answer and use only "
              "these available citation labels: "
            + allowed_source_labels
            + ". Do not turn the source labels into hyperlinks."
        )

        retry_answer = normalize_source_citations(
            run_model(retry_prompt)
        )

        # Adopt the retry only when it remains a substantive answer. If the
        # model refuses during citation repair, preserve the grounded first
        # response and repair its citation formatting deterministically.
        if (
            retry_answer
            and retry_answer != insufficient_information_message
        ):
            answer = retry_answer
        else:
            answer = initial_answer

        cited_source_numbers = (
            extract_cited_source_numbers(answer)
        )

        citation_validation_passed = bool(
            cited_source_numbers
            and cited_source_numbers.issubset(
                valid_source_numbers
            )
        )

    # --------------------------------------------------------
    # Deterministic citation fallback without discarding useful content
    # --------------------------------------------------------

    citation_fallback_used = False
    citation_repair_method = None

    if (
        answer != insufficient_information_message
        and not citation_validation_passed
    ):
        citation_fallback_used = True
        citation_repair_method = "deterministic_source_footer"

        # Remove any invalid source labels before adding only labels that map
        # to the accepted context blocks. The footer is intentionally called
        # "Retrieved evidence" rather than implying claim-level attribution.
        answer_without_invalid_citations = re.sub(
            r"\s*\[Source\s+\d+\]",
            "",
            answer
        ).strip()

        valid_labels = " ".join(
            f"[Source {source_number}]"
            for source_number in sorted(valid_source_numbers)
        )

        answer = (
            answer_without_invalid_citations
            + "\n\nRetrieved evidence: "
            + valid_labels
        )

        cited_source_numbers = extract_cited_source_numbers(answer)
        citation_validation_passed = bool(
            cited_source_numbers
            and cited_source_numbers.issubset(valid_source_numbers)
        )

    guardrail_triggered = (
        answer == insufficient_information_message
    )

    guardrail_reason = (
        "model_reported_insufficient_context"
        if guardrail_triggered
        else None
    )

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs,
        "used_docs": accepted_documents,
        "best_retrieval_score": best_retrieval_score,
        "guardrail_triggered": guardrail_triggered,
        "guardrail_reason": guardrail_reason,
        "citation_retry_performed": citation_retry_performed,
        "citation_fallback_used": citation_fallback_used,
        "citation_repair_method": citation_repair_method,
        "citation_validation_passed": (
            citation_validation_passed
        )
    }


In [ ]:
# Validate grounded generation quality and citation compliance

test_query = (
    "How do I fine-tune a transformer model?"
)

generation_retrieved_docs = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=5
)

generation_result = generate_answer(
    query=test_query,
    retrieved_docs=generation_retrieved_docs,
    generator=generator,
    max_new_tokens=300,
    minimum_retrieval_score=0.65,
    max_context_documents=2
)

print("Grounded generation test")
print("-" * 60)
print(f"Question              : {generation_result['query']}")
print(
    f"Best retrieval score  : "
    f"{generation_result['best_retrieval_score']:.4f}"
)
print(
    f"Evidence chunks used  : "
    f"{len(generation_result['used_docs'])}"
)
print(
    f"Citation retry        : "
    f"{generation_result['citation_retry_performed']}"
)
print(
    f"Citations valid       : "
    f"{generation_result['citation_validation_passed']}"
)
print(
    f"Citation fallback     : "
    f"{generation_result['citation_fallback_used']}"
)
print(
    f"Guardrail triggered   : "
    f"{generation_result['guardrail_triggered']}"
)
print("\nAnswer:")
print(generation_result["answer"])

assert generation_result["answer"].strip()
assert len(generation_result["used_docs"]) > 0
assert not generation_result["guardrail_triggered"]
assert generation_result[
    "citation_validation_passed"
]

assert re.search(
    r"\[Source\s+\d+\]",
    generation_result["answer"]
)

print("\nGrounded generation validation passed.")


In [24]:
# Diagnose retrieval confidence for an out-of-domain query

out_of_domain_query = (
    "What is the capital of France?"
)

out_of_domain_docs = retrieve_documents(
    query=out_of_domain_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=5
)

print("Out-of-domain retrieval diagnostic")
print("-" * 60)
print(f"Query: {out_of_domain_query}")

for document in out_of_domain_docs:
    print(
        f"\nRank {document['rank']} | "
        f"Score: {document['score']:.4f}"
    )
    print(f"Source: {document['source']}")
    print(
        f"Text: "
        f"{document['text'][:200]}..."
    )

highest_out_of_domain_score = max(
    document["score"]
    for document in out_of_domain_docs
)

print(
    f"\nHighest out-of-domain score: "
    f"{highest_out_of_domain_score:.4f}"
)

Out-of-domain retrieval diagnostic
------------------------------------------------------------
Query: What is the capital of France?

Rank 1 | Score: 0.6178
Source: huggingface/transformers/blob/main/README_es.md
Text: s públicos y privados.

Aquí hay algunos ejemplos:

 En procesamiento del lenguaje natural:
- [Terminación de palabras enmascaradas con BERT](https://huggingface.co/bert-base-uncased?text=Paris+is+the...

Rank 2 | Score: 0.6112
Source: huggingface/transformers/blob/main/README_zh-hans.md
Text: https://www.tensorflow.org/) — 并与之无缝整合。你可以直接使用一个框架训练你的模型然后用另一个加载和推理。

## 在线演示

你可以直接在模型页面上测试大多数 [model hub](https://huggingface.co/models) 上的模型。 我们也提供了 [私有模型托管、模型版本管理以及推理API](https://huggingface.co/pr...

Rank 3 | Score: 0.6015
Source: huggingface/course/blob/main/chapters/en/chapter7/7.mdx
Text: the special tokens to form a sentence like this:

```
[CLS] question [SEP] context [SEP]
```

Let's double-check:

```py
context = raw_datasets["train"][0]["context"]
question = raw_datas

In [25]:
# Calibrate retrieval threshold with valid domain queries

calibration_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

calibration_results = []

print("Valid-query retrieval calibration")
print("-" * 60)

for query in calibration_queries:
    documents_for_query = retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=5
    )

    best_document = documents_for_query[0]
    best_score = best_document["score"]

    calibration_results.append({
        "query": query,
        "best_score": best_score,
        "source": best_document["source"]
    })

    print(f"\nQuery      : {query}")
    print(f"Best score : {best_score:.4f}")
    print(f"Source     : {best_document['source']}")
    print(
        f"Text       : "
        f"{best_document['text'][:250]}..."
    )

print("\nCalibration summary")
print("-" * 60)

for item in calibration_results:
    print(
        f"{item['best_score']:.4f} | "
        f"{item['query']}"
    )

Valid-query retrieval calibration
------------------------------------------------------------

Query      : What is the Trainer class in transformers?
Best score : 0.7785
Source     : huggingface/course/blob/main/subtitles/en/raw/chapter3/03a_trainer-api.md
Text       : he Trainer API. The Transformers library provides a Trainer API that allows you to easily fine-tune transformer models on your own dataset. The Trainer class take your datasets, your model as well as the training hyperparameters and can perform the t...

Query      : How do I load a dataset from HuggingFace?
Best score : 0.8789
Source     : huggingface/hub-docs/blob/main/docs/hub/datasets-usage.md
Text       : Using 🤗 Datasets

Once you've found an interesting dataset on the Hugging Face Hub, you can load the dataset using 🤗 Datasets. You can click on the [**Use in dataset library** button](https://huggingface.co/datasets/samsum?library=true) to copy the c...

Query      : What is Gradio used for?
Best score : 0.7932
S

In [26]:
# Verify insufficient-information handling for an unsupported query

out_of_domain_result = generate_answer(
    query=out_of_domain_query,
    retrieved_docs=out_of_domain_docs,
    generator=generator,
    max_new_tokens=200,
    minimum_retrieval_score=0.65,
    max_context_documents=2
)

expected_guardrail_response = (
    "I don't have enough information in the provided "
    "documentation to answer this question."
)

print("Out-of-domain guardrail test")
print("-" * 60)
print(
    f"Best retrieval score : "
    f"{out_of_domain_result['best_retrieval_score']:.4f}"
)
print(
    f"Evidence chunks used : "
    f"{len(out_of_domain_result['used_docs'])}"
)
print(
    f"Guardrail triggered  : "
    f"{out_of_domain_result['guardrail_triggered']}"
)
print("\nAnswer:")
print(out_of_domain_result["answer"])

assert (
    out_of_domain_result[
        "best_retrieval_score"
    ] < 0.65
)

assert (
    out_of_domain_result[
        "guardrail_triggered"
    ]
)

assert len(
    out_of_domain_result["used_docs"]
) == 0

assert (
    out_of_domain_result["answer"]
    == expected_guardrail_response
)

print(
    "\nOut-of-domain guardrail "
    "validation passed."
)

Out-of-domain guardrail test
------------------------------------------------------------
Best retrieval score : 0.6178
Evidence chunks used : 0
Guardrail triggered  : True

Answer:
I don't have enough information in the provided documentation to answer this question.

Out-of-domain guardrail validation passed.


In [27]:
# Complete RAG pipeline with one shared configuration entry point

def rag_query(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    generator: pipeline,
    top_k: int = 5,
    max_new_tokens: int = 256,
    minimum_retrieval_score: float = 0.65,
    max_context_documents: int = 2
) -> Dict:
    """Retrieve evidence and generate one grounded answer.

    Args:
        query: Natural-language user question.
        client: Initialized Milvus client.
        collection_name: Collection containing documentation chunks.
        embedding_model: The same normalized BGE encoder used for indexing.
        generator: Hugging Face text-generation pipeline.
        top_k: Number of candidates requested from Milvus.
        max_new_tokens: Maximum answer length.
        minimum_retrieval_score: Evidence acceptance threshold.
        max_context_documents: Maximum accepted chunks passed to the LLM.

    Returns:
        Answer, evidence, citations, and guardrail diagnostics.
    """
    retrieved_docs = retrieve_documents(
        query=query,
        client=client,
        collection_name=collection_name,
        embedding_model=embedding_model,
        top_k=top_k
    )

    return generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        generator=generator,
        max_new_tokens=max_new_tokens,
        minimum_retrieval_score=minimum_retrieval_score,
        max_context_documents=max_context_documents
    )


In [28]:
# Test complete pipeline with multiple queries
test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        generator=generator,
        top_k=3,
        max_new_tokens=350
    )
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")


Q: What is the Trainer class in transformers?
A: I don't have enough information in the provided documentation to answer this question.

Q: How do I load a dataset from HuggingFace?
A: I don't have enough information in the provided documentation to answer this question.

Q: What is Gradio used for?
A: I don't have enough information in the provided documentation to answer this question.


## Stage 5: Automated Evaluation and Diagnosis

This stage separately evaluates the two failure surfaces of a RAG system:

1. **Retrieval:** a document-anchored labeled set is used to calculate precision@k and recall@k.
2. **Generation:** Opik scores answer relevance and hallucination against the exact context supplied to the LLM.

The retrieval labels are defined from canonical source paths and answer-bearing phrases before search is executed. This avoids circular evaluation in which the retriever's own outputs are treated as ground truth. The resolved chunks are printed for a human audit before metrics are calculated.


### 5.1 Labeled retrieval test set

Each test case identifies a canonical document and phrases that must appear in an answer-bearing chunk. At most three matching chunks are retained per query so the relevance set remains focused and auditable. These labels do not depend on embedding similarity or Milvus ranking.


In [29]:
# Build a deterministic, document-anchored relevance set

RETRIEVAL_LABEL_SPECS = [
    {
        "query": "What is the Trainer class in transformers?",
        "source_contains": "transformers/blob/main/docs/source/en/trainer.md",
        "required_terms": ["complete training and evaluation loop", "PyTorch"]
    },
    {
        "query": "How do I load a dataset from HuggingFace?",
        "source_contains": "datasets/blob/main/docs/source/loading.mdx",
        "required_terms": ["load_dataset", "dataset"]
    },
    {
        "query": "What is Gradio used for?",
        "source_contains": "gradio-app/gradio/blob/main/guides/",
        "required_terms": ["Gradio", "interactive machine learning"]
    },
    {
        "query": "How do I create a Hugging Face inference endpoint?",
        "source_contains": "hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx",
        "required_terms": ["Create an Endpoint", "Repository ID"]
    },
    {
        "query": "What are the benefits of fine-tuning a pretrained model?",
        "source_contains": "transformers/blob/main/docs/source/en/training.md",
        "required_terms": ["pretrained model", "fine-tun"]
    }
]


def resolve_relevant_chunk_ids(
    chunk_records: List[Dict],
    source_contains: str,
    required_terms: List[str],
    maximum_labels: int = 3
) -> List[int]:
    """Resolve answer-bearing chunks without using vector retrieval."""
    normalized_terms = [term.casefold() for term in required_terms]
    matches = []

    for chunk in chunk_records:
        source_matches = source_contains.casefold() in chunk["source"].casefold()
        text = chunk["text"].casefold()
        terms_match = all(term in text for term in normalized_terms)

        if source_matches and terms_match:
            matches.append(int(chunk["chunk_id"]))

    return matches[:maximum_labels]


retrieval_test_set = []

for specification in RETRIEVAL_LABEL_SPECS:
    relevant_ids = resolve_relevant_chunk_ids(
        chunk_records=chunks,
        source_contains=specification["source_contains"],
        required_terms=specification["required_terms"]
    )

    if not relevant_ids:
        raise ValueError(
            "No relevant chunks resolved for query: "
            f"{specification['query']}"
        )

    retrieval_test_set.append({
        "query": specification["query"],
        "relevant_chunk_ids": set(relevant_ids),
        "label_source": specification["source_contains"]
    })

print("Labeled retrieval test set")
print("-" * 60)
print(f"Queries labeled: {len(retrieval_test_set)}")

for case in retrieval_test_set:
    print(f"\nQuery        : {case['query']}")
    print(f"Relevant IDs : {sorted(case['relevant_chunk_ids'])}")
    print(f"Source rule  : {case['label_source']}")

    for chunk_id in sorted(case["relevant_chunk_ids"]):
        labeled_chunk = chunks[chunk_id]
        print(f"  - {labeled_chunk['text'][:180].replace(chr(10), ' ')}...")

assert len(retrieval_test_set) >= 5
assert all(case["relevant_chunk_ids"] for case in retrieval_test_set)


Labeled retrieval test set
------------------------------------------------------------
Queries labeled: 5

Query        : What is the Trainer class in transformers?
Relevant IDs : [20429]
Source rule  : transformers/blob/main/docs/source/en/trainer.md
  - !--Copyright 2023 The HuggingFace Team. All rights reserved.  Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with th...

Query        : How do I load a dataset from HuggingFace?
Relevant IDs : [14610, 14611, 14612]
Source rule  : datasets/blob/main/docs/source/loading.mdx
  - ne decoration-green-400 decoration-2 font-semibold" href="./nlp_load">load text dataset guide</a>.  <a id='load-from-the-hub'></a>  ## Hugging Face Hub  Datasets are loaded from a ...
  - asets import load_dataset >>> dataset = load_dataset("lhoestq/demo1") ```  Some datasets may have more than one version based on Git tags, branches, or commits. Use the `revision` ...
  - ", "test": "test.csv"} >>>

### 5.2 Precision@k and recall@k

For a ranked result list, precision@k measures how much of the first *k* results is relevant, while recall@k measures how much of the labeled relevant set appears in those results. Metrics are reported at multiple cutoffs because a result can be useful at `k=5` yet still rank the best evidence too low for a small context window.


In [30]:
# Calculate retrieval precision and recall at multiple cutoffs

def precision_at_k(
    retrieved_ids: List[int],
    relevant_ids: set,
    k: int
) -> float:
    """Return the fraction of the first k results that is relevant."""
    if k <= 0:
        raise ValueError("k must be greater than zero.")

    top_ids = retrieved_ids[:k]
    relevant_retrieved = len(set(top_ids) & relevant_ids)
    return relevant_retrieved / k


def recall_at_k(
    retrieved_ids: List[int],
    relevant_ids: set,
    k: int
) -> float:
    """Return the fraction of known-relevant chunks found in the first k."""
    if k <= 0:
        raise ValueError("k must be greater than zero.")
    if not relevant_ids:
        raise ValueError("relevant_ids cannot be empty.")

    top_ids = retrieved_ids[:k]
    relevant_retrieved = len(set(top_ids) & relevant_ids)
    return relevant_retrieved / len(relevant_ids)


# Metric unit checks with a known ranked list.
assert precision_at_k([10, 20, 30], {10, 40}, 2) == 0.5
assert recall_at_k([10, 20, 30], {10, 40}, 3) == 0.5

K_VALUES = [1, 3, 5]
retrieval_metric_rows = []

for case_number, case in enumerate(retrieval_test_set, start=1):
    ranked_documents = retrieve_documents(
        query=case["query"],
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=max(K_VALUES)
    )

    ranked_ids = [document["id"] for document in ranked_documents]

    row = {
        "case": case_number,
        "query": case["query"],
        "relevant_count": len(case["relevant_chunk_ids"]),
        "retrieved_ids": ranked_ids
    }

    for k in K_VALUES:
        row[f"precision@{k}"] = precision_at_k(
            ranked_ids,
            case["relevant_chunk_ids"],
            k
        )
        row[f"recall@{k}"] = recall_at_k(
            ranked_ids,
            case["relevant_chunk_ids"],
            k
        )

    retrieval_metric_rows.append(row)

retrieval_metrics_df = pd.DataFrame(retrieval_metric_rows)
metric_columns = [
    column
    for column in retrieval_metrics_df.columns
    if column.startswith("precision@") or column.startswith("recall@")
]

print("Retrieval evaluation by query")
print("-" * 60)
display(retrieval_metrics_df.drop(columns=["retrieved_ids"]).round(3))

macro_retrieval_metrics = retrieval_metrics_df[metric_columns].mean()

print("\nMacro-average retrieval metrics")
print("-" * 60)
for metric_name, metric_value in macro_retrieval_metrics.items():
    print(f"{metric_name:<12}: {metric_value:.3f}")

assert retrieval_metrics_df[metric_columns].apply(
    lambda column: column.between(0.0, 1.0).all()
).all()


Retrieval evaluation by query
------------------------------------------------------------


,case,query,relevant_count,precision@1,recall@1,precision@3,recall@3,precision@5,recall@5
0,1,What is the Trainer class in transformers?,1,0.0,0.000,0.333,1.000,0.2,1.000
1,2,How do I load a dataset from HuggingFace?,3,0.0,0.000,0.333,0.333,0.2,0.333
2,3,What is Gradio used for?,1,1.0,1.000,0.333,1.000,0.2,1.000
3,4,How do I create a Hugging Face inference endpo...,1,0.0,0.000,0.000,0.000,0.0,0.000
4,5,What are the benefits of fine-tuning a pretrai...,3,1.0,0.333,0.333,0.333,0.2,0.333



Macro-average retrieval metrics
------------------------------------------------------------
precision@1 : 0.400
recall@1    : 0.267
precision@3 : 0.267
recall@3    : 0.533
precision@5 : 0.160
recall@5    : 0.533


In [31]:
# Interpret retrieval metrics against explicit project thresholds

RETRIEVAL_THRESHOLDS = {
    "precision@3": 0.30,
    "recall@5": 0.80
}

retrieval_threshold_results = {
    metric_name: float(macro_retrieval_metrics[metric_name]) >= threshold
    for metric_name, threshold in RETRIEVAL_THRESHOLDS.items()
}

print("Retrieval threshold assessment")
print("-" * 60)

for metric_name, threshold in RETRIEVAL_THRESHOLDS.items():
    observed = float(macro_retrieval_metrics[metric_name])
    print(
        f"{metric_name}: observed={observed:.3f}, "
        f"target={threshold:.3f}, "
        f"status={'PASS' if observed >= threshold else 'REVIEW'}"
    )

low_retrieval_cases = retrieval_metrics_df[
    (retrieval_metrics_df["recall@5"] < RETRIEVAL_THRESHOLDS["recall@5"])
    | (retrieval_metrics_df["precision@3"] < RETRIEVAL_THRESHOLDS["precision@3"])
]

print(f"\nCases requiring diagnosis: {len(low_retrieval_cases)}")

for _, case in low_retrieval_cases.iterrows():
    print(f"\nQuery          : {case['query']}")
    print(f"Retrieved IDs  : {case['retrieved_ids']}")
    expected = next(
        item["relevant_chunk_ids"]
        for item in retrieval_test_set
        if item["query"] == case["query"]
    )
    print(f"Relevant IDs   : {sorted(expected)}")
    print(
        "Likely causes  : phrase-level labels may be too narrow, "
        "character chunks may split concepts, or dense retrieval may "
        "rank a semantically related but differently worded chunk higher."
    )


Retrieval threshold assessment
------------------------------------------------------------
precision@3: observed=0.267, target=0.300, status=REVIEW
recall@5: observed=0.533, target=0.800, status=REVIEW

Cases requiring diagnosis: 3

Query          : How do I load a dataset from HuggingFace?
Retrieved IDs  : [9211, 14610, 4087, 7680, 352]
Relevant IDs   : [14610, 14611, 14612]
Likely causes  : phrase-level labels may be too narrow, character chunks may split concepts, or dense retrieval may rank a semantically related but differently worded chunk higher.

Query          : How do I create a Hugging Face inference endpoint?
Retrieved IDs  : [16097, 3691, 356, 19289, 7783]
Relevant IDs   : [0]
Likely causes  : phrase-level labels may be too narrow, character chunks may split concepts, or dense retrieval may rank a semantically related but differently worded chunk higher.

Query          : What are the benefits of fine-tuning a pretrained model?
Retrieved IDs  : [21362, 6104, 26436, 20279,

In [32]:
# Inspect the installed Opik evaluation interfaces

import inspect
import opik

from opik.evaluation.metrics import (
    AnswerRelevance,
    Hallucination
)

from opik.evaluation.models import OpikBaseModel


print("Opik evaluation environment")
print("-" * 60)

print(
    f"Opik version                  : "
    f"{getattr(opik, '__version__', 'unknown')}"
)

print(
    f"AnswerRelevance constructor   : "
    f"{inspect.signature(AnswerRelevance)}"
)

print(
    f"Hallucination constructor     : "
    f"{inspect.signature(Hallucination)}"
)

print(
    f"AnswerRelevance.score         : "
    f"{inspect.signature(AnswerRelevance.score)}"
)

print(
    f"Hallucination.score           : "
    f"{inspect.signature(Hallucination.score)}"
)

print(
    f"OpikBaseModel abstract methods: "
    f"{OpikBaseModel.__abstractmethods__}"
)

print("\nOpik interface inspection completed.")

Opik evaluation environment
------------------------------------------------------------
Opik version                  : 2.2.77
AnswerRelevance constructor   : (model: Union[str, opik.evaluation.models.base_model.OpikBaseModel, NoneType] = None, name: str = 'answer_relevance_metric', few_shot_examples: Optional[List[opik.evaluation.metrics.llm_judges.answer_relevance.templates.FewShotExampleWithContextAnswerRelevance]] = None, few_shot_examples_no_context: Optional[List[opik.evaluation.metrics.llm_judges.answer_relevance.templates.FewShotExampleNoContextAnswerRelevance]] = None, require_context: bool = True, track: bool = True, project_name: Optional[str] = None, seed: Optional[int] = None, temperature: Optional[float] = None)
Hallucination constructor     : (model: Union[str, opik.evaluation.models.base_model.OpikBaseModel, NoneType] = None, name: str = 'hallucination_metric', few_shot_examples: Optional[List[opik.evaluation.metrics.llm_judges.hallucination.template.FewShotExampleHall

In [33]:
# Create and validate an Opik 2.2.77-compatible local Qwen judge

from typing import Any, Dict, List, Optional, Type
import json
import re

import torch
from pydantic import BaseModel
from opik.evaluation.models import OpikBaseModel


class LocalQwenOpikModel(OpikBaseModel):
    """
    Opik 2.2.77-compatible adapter for a locally loaded
    Hugging Face instruction model.
    """

    def __init__(
        self,
        generator_pipeline,
        model_name: str = "Qwen/Qwen2-1.5B-Instruct",
        max_new_tokens: int = 512
    ):
        super().__init__(model_name=model_name)

        self.generator_pipeline = generator_pipeline
        self.model = generator_pipeline.model
        self.tokenizer = generator_pipeline.tokenizer
        self.max_new_tokens = max_new_tokens

    @staticmethod
    def _clean_model_response(text: str) -> str:
        """
        Remove Markdown code fences that could prevent Opik from
        parsing the model's JSON response.
        """

        cleaned_text = text.strip()

        fenced_match = re.fullmatch(
            r"```(?:json)?\s*(.*?)\s*```",
            cleaned_text,
            flags=re.DOTALL | re.IGNORECASE
        )

        if fenced_match:
            cleaned_text = fenced_match.group(1).strip()

        return cleaned_text

    def _generate_from_messages(
        self,
        messages: List[Dict[str, Any]],
        response_format: Optional[Type[BaseModel]] = None
    ) -> str:
        """
        Generate a response from role-tagged chat messages.
        """

        prepared_messages = [
            {
                "role": message["role"],
                "content": str(message["content"])
            }
            for message in messages
        ]

        # Explicitly provide the expected JSON schema to the judge.
        if response_format is not None:
            response_schema = response_format.model_json_schema()


            # Add metric-specific score-direction instructions because
            # different Opik metrics interpret high scores differently.
            response_format_name = response_format.__name__

            if "Hallucination" in response_format_name:
                scoring_instruction = (
                    "\nHallucination scoring rules:\n"
                    "- Use score 0.0 when every claim in the answer is "
                    "supported by the supplied context.\n"
                    "- Use score 1.0 when the answer contains one or more "
                    "claims that are unsupported or contradicted by the context.\n"
                    "- A high score means hallucination is present.\n"
                    "- A low score means the answer is grounded.\n"
                    "- Do not treat factual accuracy or answer quality as a "
                    "high hallucination score.\n"
                    "- Ensure that the score and reason agree with each other.\n"
                )

            elif "AnswerRelevance" in response_format_name:
                scoring_instruction = (
                    "\nAnswer-relevance scoring rules:\n"
                    "- Use score 1.0 for a directly relevant and complete answer.\n"
                    "- Use score 0.0 for an irrelevant answer.\n"
                    "- Higher scores represent better answer relevance.\n"
                    "- Ensure that the score and reason agree with each other.\n"
                )

            else:
                scoring_instruction = (
                    "\nEnsure that the numeric score follows the metric's "
                    "definition and agrees with the written reason.\n"
                )

            schema_instruction = (
                "\n\nReturn only one valid JSON object matching the "
                "following JSON schema. Do not use Markdown code fences "
                "and do not add explanatory text.\n"
                f"{scoring_instruction}\n"
                "Required JSON schema:\n"
                f"{json.dumps(response_schema, ensure_ascii=False)}"
            )


            # Add the formatting instruction to the final user message.
            for index in range(
                len(prepared_messages) - 1,
                -1,
                -1
            ):
                if prepared_messages[index]["role"] == "user":
                    prepared_messages[index]["content"] += (
                        schema_instruction
                    )
                    break
            else:
                prepared_messages.append(
                    {
                        "role": "user",
                        "content": schema_instruction
                    }
                )

        prompt = self.tokenizer.apply_chat_template(
            prepared_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3072
        )

        model_inputs = {
            key: value.to(self.model.device)
            for key, value in model_inputs.items()
        }

        input_token_count = model_inputs[
            "input_ids"
        ].shape[1]

        with torch.inference_mode():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id
            )

        response_ids = generated_ids[
            0,
            input_token_count:
        ]

        generated_text = self.tokenizer.decode(
            response_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        return self._clean_model_response(
            generated_text
        )

    def generate_chat_completion(
        self,
        messages: List[Dict[str, Any]],
        response_format: Optional[Type[BaseModel]] = None,
        **kwargs: Any
    ) -> Dict[str, str]:
        """
        Interface used by Opik 2.2.77 LLM-as-judge metrics.
        """

        generated_text = self._generate_from_messages(
            messages=messages,
            response_format=response_format
        )

        return {
            "role": "assistant",
            "content": generated_text
        }

    def generate_string(
        self,
        input: str,
        response_format: Optional[Type[BaseModel]] = None,
        **kwargs: Any
    ) -> str:
        """
        Simplified string-generation interface required by
        OpikBaseModel.
        """

        messages = [
            {
                "role": "system",
                "content": (
                    "You are an objective RAG evaluation judge. "
                    "Follow the requested response format exactly."
                )
            },
            {
                "role": "user",
                "content": input
            }
        ]

        response = self.generate_chat_completion(
            messages=messages,
            response_format=response_format,
            **kwargs
        )

        return response["content"]

    def generate_provider_response(
        self,
        messages: List[Dict[str, Any]],
        **kwargs: Any
    ) -> Dict[str, Any]:
        """
        OpenAI-style provider response required by
        OpikBaseModel.
        """

        response_format = kwargs.pop(
            "response_format",
            None
        )

        message = self.generate_chat_completion(
            messages=messages,
            response_format=response_format,
            **kwargs
        )

        return {
            "id": "local-qwen-opik-response",
            "object": "chat.completion",
            "model": self.model_name,
            "choices": [
                {
                    "index": 0,
                    "message": message,
                    "finish_reason": "stop"
                }
            ]
        }


# ------------------------------------------------------------
# Initialize the corrected local Opik judge
# ------------------------------------------------------------

opik_judge_model = LocalQwenOpikModel(
    generator_pipeline=generator,
    model_name="Qwen/Qwen2-1.5B-Instruct",
    max_new_tokens=512
)


# ------------------------------------------------------------
# Validate string generation
# ------------------------------------------------------------

adapter_test_response = opik_judge_model.generate_string(
    'Return only this JSON object: '
    '{"status": "adapter_ready"}'
)


# ------------------------------------------------------------
# Validate chat-completion generation
# ------------------------------------------------------------

chat_test_response = (
    opik_judge_model.generate_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "Return the requested JSON exactly."
                )
            },
            {
                "role": "user",
                "content": (
                    'Return {"status": "chat_ready"}'
                )
            }
        ]
    )
)


print("Local Opik judge validation")
print("-" * 60)
print(f"Model name      : {opik_judge_model.model_name}")
print(f"Model device    : {opik_judge_model.model.device}")
print(f"Maximum tokens  : {opik_judge_model.max_new_tokens}")
print(f"String response : {adapter_test_response}")
print(f"Chat response   : {chat_test_response}")


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert isinstance(
    adapter_test_response,
    str
), "generate_string() must return a string."

assert isinstance(
    chat_test_response,
    dict
), "generate_chat_completion() must return a dictionary."

assert chat_test_response.get(
    "role"
) == "assistant", (
    "The chat response role must be assistant."
)

assert isinstance(
    chat_test_response.get("content"),
    str
), "The chat response content must be a string."

assert len(
    chat_test_response["content"].strip()
) > 0, "The chat response content is empty."

print(
    "\nOpik 2.2.77-compatible local judge "
    "validation passed."
)

Local Opik judge validation
------------------------------------------------------------
Model name      : Qwen/Qwen2-1.5B-Instruct
Model device    : cuda:0
Maximum tokens  : 512
String response : {"status": "adapter_ready"}
Chat response   : {'role': 'assistant', 'content': '{"status": "chat_ready"}'}

Opik 2.2.77-compatible local judge validation passed.


In [34]:
# Validate Opik relevance and hallucination metrics on a grounded example

import numpy as np

from opik.evaluation.metrics import (
    AnswerRelevance,
    Hallucination
)


# Local evaluation avoids external Opik tracking or API requirements
answer_relevance_metric = AnswerRelevance(
    model=opik_judge_model,
    require_context=True,
    track=False,
    temperature=0.0,
    seed=42
)

hallucination_metric = Hallucination(
    model=opik_judge_model,
    track=False,
    temperature=0.0,
    seed=42
)


# ------------------------------------------------------------
# Controlled grounded example
# ------------------------------------------------------------

evaluation_question = (
    "What is the Trainer class in Transformers?"
)

evaluation_answer = (
    "The Trainer class provides a complete training and "
    "evaluation loop for PyTorch models. It accepts a model, "
    "datasets, training arguments, and an optional evaluation "
    "function."
)

evaluation_context = [
    (
        "The Transformers Trainer class provides a complete "
        "training and evaluation loop for PyTorch models. "
        "It takes the model, training arguments, training and "
        "evaluation datasets, and an optional evaluation function."
    )
]


print("Running Opik metric validation...")
print("-" * 60)


# ------------------------------------------------------------
# Calculate both Opik metrics
# ------------------------------------------------------------

relevance_result = answer_relevance_metric.score(
    input=evaluation_question,
    output=evaluation_answer,
    context=evaluation_context
)

hallucination_result = hallucination_metric.score(
    input=evaluation_question,
    output=evaluation_answer,
    context=evaluation_context
)


# ------------------------------------------------------------
# Report metric results
# ------------------------------------------------------------

print("\nOpik metric validation")
print("-" * 60)
print(f"Question              : {evaluation_question}")
print(f"Answer relevance score: {relevance_result.value}")
print(f"Relevance failed      : {relevance_result.scoring_failed}")
print(f"Relevance reason      : {relevance_result.reason}")
print()
print(f"Hallucination score   : {hallucination_result.value}")
print(f"Hallucination failed  : {hallucination_result.scoring_failed}")
print(f"Hallucination reason  : {hallucination_result.reason}")


# ------------------------------------------------------------
# Validate metric integrity
# ------------------------------------------------------------

assert not relevance_result.scoring_failed, (
    "AnswerRelevance failed to parse the judge response."
)

assert not hallucination_result.scoring_failed, (
    "Hallucination failed to parse the judge response."
)

assert np.isfinite(relevance_result.value), (
    "Answer relevance returned an invalid score."
)

assert np.isfinite(hallucination_result.value), (
    "Hallucination returned an invalid score."
)

assert 0.0 <= relevance_result.value <= 1.0, (
    "Answer relevance must be between 0 and 1."
)

assert 0.0 <= hallucination_result.value <= 1.0, (
    "Hallucination must be between 0 and 1."
)

print("\nOpik metric validation passed.")

Running Opik metric validation...
------------------------------------------------------------

Opik metric validation
------------------------------------------------------------
Question              : What is the Trainer class in Transformers?
Answer relevance score: 1.0
Relevance failed      : False
Relevance reason      : The provided answer directly addresses the question about the Trainer class in Transformers by explaining its purpose and components. It also correctly uses the Trainer class to provide a complete training and evaluation loop for PyTorch models.

Hallucination score   : 0.0
Hallucination failed  : False
Hallucination reason  : ['The context does not mention anything about the Trainer class in Transformers.']

Opik metric validation passed.


### 5.3 Judge calibration and limitations

An LLM judge is itself a model and can produce inconsistent scores. Before interpreting pipeline results, the same Opik hallucination metric is tested on a supported answer and a deliberately unsupported answer. If the ordering is wrong, the notebook reports the Opik numbers but flags them as unreliable rather than presenting them as conclusive evidence.


In [35]:
# Sanity-check score direction with a deliberately unsupported answer

unsupported_evaluation_answer = (
    "The Trainer class automatically deploys every trained model to a "
    "production endpoint and guarantees 99.99 percent accuracy."
)

unsupported_hallucination_result = hallucination_metric.score(
    input=evaluation_question,
    output=unsupported_evaluation_answer,
    context=evaluation_context
)

judge_calibration_passed = bool(
    not hallucination_result.scoring_failed
    and not unsupported_hallucination_result.scoring_failed
    and unsupported_hallucination_result.value > hallucination_result.value
)

print("Opik judge calibration")
print("-" * 60)
print(f"Grounded answer score   : {hallucination_result.value:.3f}")
print(
    f"Unsupported answer score: "
    f"{unsupported_hallucination_result.value:.3f}"
)
print(f"Calibration passed      : {judge_calibration_passed}")

if not judge_calibration_passed:
    print(
        "WARNING: The local 1.5B judge did not rank the deliberately "
        "unsupported answer as more hallucinatory. Treat subsequent "
        "Opik hallucination scores as diagnostic only and use a stronger "
        "judge before production deployment."
    )


Opik judge calibration
------------------------------------------------------------
Grounded answer score   : 0.000
Unsupported answer score: 1.000
Calibration passed      : True


In [ ]:
# Evaluate the complete RAG pipeline with Opik metrics

import time
import numpy as np
import pandas as pd


evaluation_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?"
]

opik_evaluation_results = []


print("Starting end-to-end RAG evaluation")
print("-" * 60)
print(f"Evaluation queries : {len(evaluation_queries)}")
print(f"Metrics            : AnswerRelevance, Hallucination")


for case_number, query in enumerate(
    evaluation_queries,
    start=1
):
    print(
        f"\nEvaluating case "
        f"{case_number}/{len(evaluation_queries)}"
    )
    print(f"Query: {query}")

    case_start_time = time.perf_counter()

    # --------------------------------------------------------
    # Execute the complete retrieval and generation pipeline
    # --------------------------------------------------------

    rag_result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        generator=generator,
        top_k=3,
        max_new_tokens=350
    )

    used_documents = rag_result["used_docs"]

    # Include source metadata with the context so evidence-source
    # lines in the generated answer remain verifiable.
    evaluation_contexts = [
        (
            f"Source: {document['source']}\n"
            f"{document['text']}"
        )
        for document in used_documents
    ]

    # --------------------------------------------------------
    # Calculate Opik metrics
    # --------------------------------------------------------

    relevance_score_result = (
        answer_relevance_metric.score(
            input=query,
            output=rag_result["answer"],
            context=evaluation_contexts
        )
    )

    hallucination_score_result = (
        hallucination_metric.score(
            input=query,
            output=rag_result["answer"],
            context=evaluation_contexts
        )
    )

    elapsed_seconds = (
        time.perf_counter() - case_start_time
    )

    scoring_failed = bool(
        relevance_score_result.scoring_failed
        or hallucination_score_result.scoring_failed
    )

    opik_evaluation_results.append(
        {
            "case": case_number,
            "query": query,
            "best_retrieval_score": (
                rag_result["best_retrieval_score"]
            ),
            "evidence_chunks": len(used_documents),
            "answer_relevance": (
                relevance_score_result.value
            ),
            "hallucination": (
                hallucination_score_result.value
            ),
            "relevance_reason": (
                relevance_score_result.reason
            ),
            "hallucination_reason": (
                hallucination_score_result.reason
            ),
            "citation_valid": (
                rag_result[
                    "citation_validation_passed"
                ]
            ),
            "citation_fallback_used": (
                rag_result["citation_fallback_used"]
            ),
            "guardrail_triggered": (
                rag_result["guardrail_triggered"]
            ),
            "scoring_failed": scoring_failed,
            "elapsed_seconds": elapsed_seconds,
            "answer": rag_result["answer"]
        }
    )

    print(
        f"Retrieval score    : "
        f"{rag_result['best_retrieval_score']:.4f}"
    )
    print(
        f"Evidence chunks    : "
        f"{len(used_documents)}"
    )
    print(
        f"Answer relevance   : "
        f"{relevance_score_result.value:.4f}"
    )
    print(
        f"Hallucination      : "
        f"{hallucination_score_result.value:.4f}"
    )
    print(
        f"Citations valid    : "
        f"{rag_result['citation_validation_passed']}"
    )
    print(
        f"Metric failure     : "
        f"{scoring_failed}"
    )
    print(
        f"Processing time    : "
        f"{elapsed_seconds:.2f} seconds"
    )


# ------------------------------------------------------------
# Create an evaluation summary table
# ------------------------------------------------------------

opik_evaluation_df = pd.DataFrame(
    opik_evaluation_results
)

summary_columns = [
    "case",
    "query",
    "best_retrieval_score",
    "evidence_chunks",
    "answer_relevance",
    "hallucination",
    "citation_valid",
    "citation_fallback_used",
    "scoring_failed",
    "elapsed_seconds"
]

print("\nEnd-to-end Opik evaluation summary")
print("-" * 60)

display(
    opik_evaluation_df[
        summary_columns
    ].round(
        {
            "best_retrieval_score": 4,
            "answer_relevance": 4,
            "hallucination": 4,
            "elapsed_seconds": 2
        }
    )
)


# ------------------------------------------------------------
# Aggregate metrics
# ------------------------------------------------------------

average_relevance = (
    opik_evaluation_df[
        "answer_relevance"
    ].mean()
)

average_hallucination = (
    opik_evaluation_df[
        "hallucination"
    ].mean()
)

citation_pass_rate = (
    opik_evaluation_df[
        "citation_valid"
    ].mean()
)

citation_fallback_rate = (
    opik_evaluation_df[
        "citation_fallback_used"
    ].mean()
)

metric_failure_count = int(
    opik_evaluation_df[
        "scoring_failed"
    ].sum()
)

average_latency = (
    opik_evaluation_df[
        "elapsed_seconds"
    ].mean()
)


print("\nAggregate evaluation metrics")
print("-" * 60)
print(
    f"Average answer relevance : "
    f"{average_relevance:.4f}"
)
print(
    f"Average hallucination    : "
    f"{average_hallucination:.4f}"
)
print(
    f"Citation validation rate : "
    f"{citation_pass_rate:.2%}"
)
print(
    f"Citation fallback rate   : "
    f"{citation_fallback_rate:.2%}"
)
print(
    f"Metric failures          : "
    f"{metric_failure_count}"
)
print(
    f"Average pipeline latency : "
    f"{average_latency:.2f} seconds"
)


# ------------------------------------------------------------
# Evaluation integrity checks
# ------------------------------------------------------------

assert len(opik_evaluation_df) == len(
    evaluation_queries
), "Not all evaluation queries were processed."

assert metric_failure_count == 0, (
    "One or more Opik metric calculations failed."
)

assert opik_evaluation_df[
    "answer_relevance"
].between(
    0.0,
    1.0
).all(), "Invalid relevance score detected."

assert opik_evaluation_df[
    "hallucination"
].between(
    0.0,
    1.0
).all(), "Invalid hallucination score detected."

if not opik_evaluation_df["citation_valid"].all():
    print(
        "WARNING: At least one answer failed citation validation; "
        "review the generation diagnostics below."
    )

print(
    "\nEnd-to-end Opik evaluation completed successfully."
)


### 5.4 Threshold interpretation and failure diagnosis

Retrieval and generation results are compared with explicit project thresholds. Low-recall cases are inspected as retrieval or chunking failures; answers with adequate evidence but weak relevance are treated as generation failures; and hallucination results are trusted only when the judge calibration passes.


In [37]:
# Diagnose evaluation cases with high hallucination scores

HALLUCINATION_FAILURE_THRESHOLD = 0.5

high_hallucination_cases = [
    evaluation_record
    for evaluation_record in opik_evaluation_results
    if evaluation_record["hallucination"]
    >= HALLUCINATION_FAILURE_THRESHOLD
]


print("High-hallucination case diagnosis")
print("-" * 60)

print(
    f"Failure threshold : "
    f"{HALLUCINATION_FAILURE_THRESHOLD:.2f}"
)

print(
    f"Cases detected    : "
    f"{len(high_hallucination_cases)}"
)


for evaluation_record in high_hallucination_cases:
    query = evaluation_record["query"]

    # Retrieve the same ranked evidence used by the RAG pipeline
    diagnostic_documents = retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=3
    )

    accepted_documents = [
        document
        for document in diagnostic_documents
        if document["score"] >= 0.65
    ][:2]

    print("\n" + "=" * 60)

    print(
        f"Case                 : "
        f"{evaluation_record['case']}"
    )

    print(
        f"Query                : "
        f"{query}"
    )

    print(
        f"Answer relevance     : "
        f"{evaluation_record['answer_relevance']:.4f}"
    )

    print(
        f"Hallucination score  : "
        f"{evaluation_record['hallucination']:.4f}"
    )

    print(
        f"Hallucination reason : "
        f"{evaluation_record['hallucination_reason']}"
    )

    print("\nGenerated answer:")
    print(evaluation_record["answer"])

    print("\nRetrieved evidence:")

    for source_number, document in enumerate(
        accepted_documents,
        start=1
    ):
        print(
            f"\n[Source {source_number}]"
        )

        print(
            f"Similarity score: "
            f"{document['score']:.4f}"
        )

        print(
            f"Source path     : "
            f"{document['source']}"
        )

        print("Context:")
        print(
            document["text"][:1500]
        )


if not high_hallucination_cases:
    print(
        "\nNo cases exceeded the hallucination "
        "failure threshold."
    )

print("\nHallucination failure diagnosis completed.")

High-hallucination case diagnosis
------------------------------------------------------------
Failure threshold : 0.50
Cases detected    : 0

No cases exceeded the hallucination failure threshold.

Hallucination failure diagnosis completed.


In [38]:
# Interpret generation metrics against explicit project thresholds

GENERATION_THRESHOLDS = {
    "minimum_answer_relevance": 0.80,
    "maximum_hallucination": 0.20,
    "minimum_citation_rate": 1.00
}

generation_threshold_results = {
    "answer_relevance": average_relevance
    >= GENERATION_THRESHOLDS["minimum_answer_relevance"],
    "hallucination": average_hallucination
    <= GENERATION_THRESHOLDS["maximum_hallucination"],
    "citation_rate": citation_pass_rate
    >= GENERATION_THRESHOLDS["minimum_citation_rate"]
}

print("Generation threshold assessment")
print("-" * 60)
print(
    f"Answer relevance : {average_relevance:.3f} "
    f"(target >= {GENERATION_THRESHOLDS['minimum_answer_relevance']:.2f})"
)
print(
    f"Hallucination    : {average_hallucination:.3f} "
    f"(target <= {GENERATION_THRESHOLDS['maximum_hallucination']:.2f})"
)
print(
    f"Citation rate    : {citation_pass_rate:.1%} "
    f"(target >= {GENERATION_THRESHOLDS['minimum_citation_rate']:.0%})"
)
print(f"Judge calibrated : {judge_calibration_passed}")

if not generation_threshold_results["answer_relevance"]:
    print(
        "Diagnosis: low relevance with adequate retrieval points to the "
        "generation prompt/model; low relevance with weak recall points "
        "first to retrieval or chunking."
    )

if not generation_threshold_results["hallucination"]:
    print(
        "Diagnosis: inspect each claim against used_docs. Possible causes "
        "include an underspecified prompt, irrelevant context, or a weak "
        "judge. Judge calibration must pass before this metric is trusted."
    )


Generation threshold assessment
------------------------------------------------------------
Answer relevance : 0.700 (target >= 0.80)
Hallucination    : 0.000 (target <= 0.20)
Citation rate    : 0.0% (target >= 100%)
Judge calibrated : True
Diagnosis: low relevance with adequate retrieval points to the generation prompt/model; low relevance with weak recall points first to retrieval or chunking.


## Stage 6: Rigor, Trade-offs, and Reproducibility

### 6.1 Pipeline organization and checkpoints

The notebook follows the dependency order required for a clean run: environment setup, data inspection, chunking, embedding, storage, retrieval, grounded generation, retrieval evaluation, generation evaluation, and diagnosis. Each stage prints its key checkpoint: document and chunk counts, embedding dimension and norms, inserted row count, ranked retrieval scores, answer evidence, quality metrics, and latency.

### 6.2 Design choices and trade-offs

- **Chunking:** A 1,000-character window with 200-character overlap gives an 800-character stride. It is inexpensive and deterministic, and the overlap retains boundary context. Its limitation is that a character boundary can split a sentence or code block. A production iteration should compare sentence-aware or token-aware chunking using the same labeled retrieval set.
- **Embedding model:** `BAAI/bge-small-en-v1.5` produces compact 384-dimensional vectors and offers a practical quality, memory, and latency balance. A larger embedding model may improve recall but increases indexing time, storage, and query latency.
- **Normalization and metric:** L2-normalized document and query vectors allow Milvus inner product to produce cosine-equivalent ranking. Mixing normalized documents with an unnormalized query would invalidate score calibration.
- **Vector database:** Milvus Lite gives a persistent Milvus-compatible local workflow without operating a server. The collection is dropped and recreated on rerun to prevent duplicate IDs; this is idempotent but rebuilds the local index.
- **Generator:** `Qwen/Qwen2-1.5B-Instruct` fits common Colab GPUs and runs locally without a paid generation API. The trade-off is weaker instruction following and grounded synthesis than larger models.
- **Context policy:** Only the two highest-ranked chunks above a calibrated score threshold are supplied to the LLM. This controls latency and distractor context, but may omit complementary evidence for multi-part questions.
- **Guardrails:** Unsupported queries still fail closed when retrieval confidence is insufficient. Citation-format failures preserve the useful answer and add an auditable footer containing only accepted source labels; the fallback rate is reported separately because citation syntax does not prove claim-level grounding.
- **Evaluation:** Document-anchored retrieval labels avoid circular scoring. Opik provides reproducible interfaces for relevance and hallucination, while the calibration test explicitly exposes the reliability limit of a small local LLM judge.

### 6.3 Secure and reproducible execution

- No token or credential is hardcoded. An optional Hugging Face token is read from `HF_TOKEN`.
- Dependency ranges are bounded and the exact runtime versions are printed below.
- Random seeds and deterministic decoding are used where supported.
- Artifact paths work in both Colab and a repository-local Jupyter session.
- The full dataset is used; no hidden fixed subset is required for indexing.
- The final PDF must be produced only after a fresh **Restart runtime → Run all** execution.


In [39]:
# Print a final reproducibility manifest and acceptance summary

PACKAGE_NAMES = [
    "datasets",
    "sentence-transformers",
    "transformers",
    "pymilvus",
    "milvus-lite",
    "opik",
    "torch",
    "numpy",
    "pandas"
]

runtime_versions = {}
for package_name in PACKAGE_NAMES:
    try:
        runtime_versions[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        runtime_versions[package_name] = "not installed"

reproducibility_manifest = {
    "random_seed": RANDOM_SEED,
    "device": DEVICE,
    "dataset": DATASET_NAME,
    "documents": len(documents),
    "chunks": len(chunks),
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIM,
    "vectors_normalized": True,
    "vector_metric": "IP",
    "records_inserted": stored_row_count,
    "generation_model": LLM_MODEL_NAME,
    "hf_token_configured": HF_TOKEN is not None,
    "runtime_versions": runtime_versions
}

print("Final reproducibility manifest")
print("-" * 60)
print(json.dumps(reproducibility_manifest, indent=2))

print("\nAcceptance summary")
print("-" * 60)
print(f"Retrieval thresholds : {retrieval_threshold_results}")
print(f"Generation thresholds: {generation_threshold_results}")
print(f"Judge calibration    : {judge_calibration_passed}")
print(f"Metric failures      : {metric_failure_count}")


Final reproducibility manifest
------------------------------------------------------------
{
  "random_seed": 42,
  "device": "cuda",
  "dataset": "m-ric/huggingface_doc",
  "documents": 2647,
  "chunks": 27434,
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "embedding_dimension": 384,
  "vectors_normalized": true,
  "vector_metric": "IP",
  "records_inserted": 27434,
  "generation_model": "Qwen/Qwen2-1.5B-Instruct",
  "hf_token_configured": false,
  "runtime_versions": {
    "datasets": "4.8.5",
    "sentence-transformers": "5.7.0",
    "transformers": "5.17.0",
    "pymilvus": "2.6.17",
    "milvus-lite": "2.5.1",
    "opik": "2.2.77",
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "pandas": "2.2.3"
  }
}

Acceptance summary
------------------------------------------------------------
Retrieval thresholds : {'precision@3': False, 'recall@5': False}
Generation thresholds: {'answer_relevance': np.False_, 'hallucination': np.True_

## Final Submission Checklist

1. Restart the Colab runtime and run every cell from top to bottom.
2. Confirm that all assertions pass and that no cell contains an error traceback.
3. Review the labeled retrieval test set, retrieval metrics, Opik results,
   calibration result, and diagnostic cases.
4. If the local Qwen judge calibration does not pass, disclose that its
   hallucination scores are diagnostic only and should be verified with a
   stronger judge before production use.
5. Save the completed notebook in Google Colab.
6. Use **File → Download → Download .ipynb**.
7. Upload the downloaded `.ipynb` file to the Scaler Business Case platform.

The thresholds in this notebook are proof-of-concept acceptance targets.
They should be recalibrated with a larger human-reviewed evaluation set
before production deployment.
